# Qwen3-ASR on Kabyle Data
Ce notebook permet de tester le nouveau modèle **Qwen3-ASR** (All-in-One Speech Recognition) d'Alibaba sur vos données audio Kabyle.
Il est configuré pour fonctionner directement sur Google Colab pour bénéficier de la puissance du GPU T4 intégré sans devoir l'installer localement.

In [ ]:
# Étape 1 : Installation des dépendances requises par Qwen3-ASR
!pip install -U qwen-asr
!pip install -U flash-attn --no-build-isolation

In [ ]:
# Étape 2 : Connecter Google Drive pour accéder à vos fichiers audio locaux
from google.colab import drive
drive.mount('/content/drive')

# Assurez-vous d'avoir uploadé vos audios kabyles quelque part dans votre Drive,
# puis modifiez le chemin ci-dessous dans l'étape 4.

In [ ]:
# Étape 3 : Chargement du modèle Qwen3-ASR
import torch
from qwen_asr import Qwen3ASRModel

print("Chargement de Qwen3-ASR-1.7B en cours (cela peut prendre quelques minutes)...")
model = Qwen3ASRModel.from_pretrained(
    "Qwen/Qwen3-ASR-1.7B", 
    dtype=torch.bfloat16, 
    device_map="cuda:0",
    # Décommentez la ligne ci-dessous si Flash Attention 2 s'est bien installé (améliore grandement la vitesse)
    # attn_implementation="flash_attention_2",
    max_inference_batch_size=32,
)
print("Modèle chargé avec succès !")

In [ ]:
# Étape 4 : Tester la transcription sur vos fichiers audio
import glob

# TODO: Modifiez ce chemin pour pointer vers le dossier contenant vos audios sur votre Google Drive
# Par exemple : '/content/drive/MyDrive/gestion_appels_telephoniques/dataset/appels_valides/'
AUDIO_DIR = '/content/drive/MyDrive/'
audio_files = glob.glob(f"{AUDIO_DIR}/**/*.wav", recursive=True) + glob.glob(f"{AUDIO_DIR}/**/*.mp3", recursive=True)

print(f"Trouvé {len(audio_files)} fichiers audio.")

# On teste l'inférence sur les 5 premiers fichiers pour l'instant
test_files = audio_files[:5]

if test_files:
    print("\nDémarrage de la transcription...")
    results = model.transcribe(
        audio=test_files,
        language=None, # Laisse le modèle détecter la langue lui-même
    )
    
    for f, r in zip(test_files, results):
        print(f"\n--- Fichier : {f.split('/')[-1]} ---")
        print(f"Langue détectée : {r.language}")
        print(f"Transcription   : {r.text}")
else:
    print("Aucun fichier audio n'a été trouvé. Vérifiez que le chemin AUDIO_DIR est correct dans votre Google Drive.")